In [ ]:
#1-1 CSV 파일 읽기
import pandas as pd

df_house = pd.read_csv("/content/1.01.house.price.csv") # 파일 경로 보고 읽기

print(df_house.head(1)) # 첫번째 행 출력하기

print(len(df_house)) # 전체 데이터의 행 수

print(type(df_house["price"])) # 파일의 자료형 출력


   ID  size  year  price
0   1    34     5    5.0
10
<class 'pandas.core.series.Series'>


In [ ]:
#1-1 CSV 파일 없을때 오류코드
import pandas as pd # 오류

df_house = pd.read_csv("/content/1.01.House.price.csv") # 파일 경로 보고 읽기

print(df_house.head(1)) # 첫번째 행 출력하기

print(len(df_house)) # 전체 데이터의 행 수

print(type(df_house["price"])) # 파일의 자료형 출력


FileNotFoundError: [Errno 2] No such file or directory: '/content/1.01.House.price.csv'

In [ ]:
#1-2 문자열을 숫자로 변환하기

houses = [] #1-2
excluded_count = 0

for index, row in df_house.iterrows(): # 반복문 순회
  if row[["ID","size","year","price"]].isna().any():
     print(f"인덱스{index} 제외 : 필수 값 누락") # 출력
     excluded_count += 1 #누락 시 카운트 추가
     continue


  try: #houses 에 추가 및 오류가 있을시 건너뛰기 위함
    houses.append({
        "ID": int(row["ID"]),
        "size": int(row["size"]),
        "year": int(row["year"]),
        "price": float(row["price"])
    })

  except(ValueError,TypeError,OverflowError): # 오류 있을시 카운트 1 추가
   print(f"{index} 제외: 숫자 변환 불가")
   excluded_count += 1
   continue


print(f"정상 저장 행 수: {len(houses)}")
print(f"제외 행 수: {excluded_count}")

정상 저장 행 수: 10
제외 행 수: 0


In [ ]:
#1-3 조건에 맞는 주택 찾기

houses_test = houses.copy()

houses_test.append({"ID": 101, "year": 5, "price": 3})  # size 키 없음
houses_test.append({"ID": 102, "size": 35, "year": 5, "price": None})  # 값 없음
houses_test.append({"ID": 103, "size": "오류", "year": 5, "price": 3})  # 숫자로 변환할 수 없음
houses_test.append({"ID": 104, "size": 35, "year": 5, "price": 3})  # 정상 데이터

selected = []

# size 또는 price 키가 누락 되었을때 for 문 if 문
for house in houses_test:
  if "size" not in house or "price" not in house:
    print(f"{house["ID"]} 제외: size 또는 price 키 누락")
    continue

# size 또는 price 키가 None 일 때 for 문 if 문
  if house["size"] is None or house["price"] is None:
      print(f"{house["ID"]} 제외: size 또는 price가 None")
      continue

# size 또는 price 키가 결측값 일 때 for 문 if 문
  if pd.isna(house["size"]) or pd.isna(house["price"]):
      print(f"{house["ID"]} 제외: size 또는 price가 결측값")
      continue

# 키 누락,None,결측값,숫자변환 제외 사유를 출력하고 건너 뛸때
  try :
    size = float(house["size"])
    price = float(house["price"])
  except(ValueError,TypeError,OverflowError):
      print(f"{house["ID"]} 제외: 숫자 변환 불가")
      continue
  if pd.isna(size) or pd.isna(price):
      print(f"{house["ID"]} 제외: 숫자 변환 후 결측값")
      continue
  if size <= 0 or price < 0 :
      print(f"{house["ID"]} 제외 : size는 양수, price는 0 이상이어야 함.")
      continue
  if size >= 30 and price <= 4:
    selected.append(house)

for house in selected:
  print(f"선택된 주택 ID: {house["ID"]}")

print(f"선택된 주택의 전체 건수: {len(selected)}")




101 제외: size 또는 price 키 누락
102 제외: size 또는 price가 None
103 제외: 숫자 변환 불가
선택된 주택 ID: 3
선택된 주택 ID: 4
선택된 주택 ID: 5
선택된 주택 ID: 9
선택된 주택 ID: 10
선택된 주택 ID: 104
선택된 주택의 전체 건수: 6


In [ ]:
#1-4 함수와 예외 처리

# 함수 정의
def unit_price(price, size):

  try:

    price = float(price)
    size = float(size)

    # 값 오류 일때 안내 문구
  except ValueError:

    print("값 오류: 숫자로 변환할 수 없는 문자열입니다.")
    return None

    # 자료형 오류 일때 안내 문구
  except TypeError:
    print("자료형 오류: 숫자 또는 숫자 형태의 문자열을 입력하세요.")
    return None
# size와 price 조건
  if size <= 0:
    print("오류: size는 0보다 커야합니다.")
    return None
  if price < 0:
    print("오류: price는 음수여서는 안됩니다.")
    return None

  return price / size

print(unit_price(4, 40))
print(unit_price("4", "40"))
print(unit_price(4, 0))
print(unit_price(-4, 40))
print(unit_price("오류",40))
print(unit_price(None,40))
print(unit_price(6,30))


0.1
0.1
오류: size는 0보다 커야합니다.
None
오류: price는 음수여서는 안됩니다.
None
값 오류: 숫자로 변환할 수 없는 문자열입니다.
None
자료형 오류: 숫자 또는 숫자 형태의 문자열을 입력하세요.
None
0.2


In [ ]:
#1-5 목적에 맞는 자료 구조 선택하기
house_id = []

for house in houses:
  house_id.append(house["ID"]) # list 선택 이유 : 원래 순서대로 저장하고 하나씩 추가 할 수 있기 때문이다.

year = set()

for house in houses:
  year.add(house["year"]) # 집합: 중복 없이 저장하기에는 집합이 좋기 때문이다.

price_by_id = {}

for house in houses:
  price_by_id[house["ID"]] = house["price"] # 딕셔너리 : ID로 가격을 조회하기 좋기 때문이다.

conditions = (30,4) # 튜플은 추가/삭제 교체 등을 할 수 없기 때문이다.

print(price_by_id[3])
print(house_id)
print(year)
print(conditions)

4.0
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
{2, 5, 12, 18, 19, 20, 22}
(30, 4)
